# Phase 5: Web search (Context Check agent)

Is a reported problem already a known or planned outage? The Context Check agent searches the web with Tavily and answers with cited sources.

It only receives the service, area and municipality, never the report text, so no personal details can reach a search. Results come from the live web, so yours will differ.

In [ ]:
from community_voice.agents.context_check import run_context_check
from community_voice.tools.web_search import web_search

## 1. A raw search

The tool returns the last month's results as title, URL and a short snippet.

In [ ]:
print(web_search.invoke({"query": "Soweto power outage Johannesburg"}))

## 2. Check an area

`show` prints each search the agent makes (ACT), what came back (RESULT, shortened), and the final answer.

In [ ]:
def show(state):
    for message in state["messages"]:
        if message.type == "ai" and message.tool_calls:
            for call in message.tool_calls:
                if call["name"] != "ContextCheckResult":
                    print("ACT   ", call["name"], call["args"])
        elif message.type == "tool" and message.name != "ContextCheckResult":
            print("RESULT", message.text[:150].replace("\n", " "))
    print(state["structured_response"].model_dump_json(indent=2))

In [ ]:
show(run_context_check("electricity", "Soweto", "Johannesburg"))

In [ ]:
show(run_context_check("water", "Soshanguve", "Tshwane"))

## 3. An area with nothing to find

The area is made up. The agent should say `nothing_found`, with low confidence and no sources, instead of inventing a link.

In [ ]:
show(run_context_check("water", "Zzyzx Heights", "eThekwini"))

## What to notice

- The agent chose its own search words and could search again.
- Every source is a page the search returned.
- With no evidence, it said so.
- Web text is untrusted: the prompt tells the agent to treat it as data.